In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 1
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

[WARNING 06-25 16:24:43] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
[WARNING 06-25 16:25:23] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


Trial 0 =========================================
14.346707921289035



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
[WARNING 06-25 16:25:40] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


Trial 1 =========================================
15.678620139789974



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
[WARNING 06-25 16:26:06] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


...

Trial 97 =========================================
15.929222010399386



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))
[WARNING 06-25 17:05:35] ax.modelbridge.transforms.standardize_y: Outcome t1 is constant, within tolerance.


Trial 98 =========================================
14.44567619702977



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/ax/modelbridge/cross_validation.py:382: UserWarning: Encountered exception in computing model fit quality: Outcome `t1` was not observed.
  warn("Encountered exception in computing model fit quality: " + str(e))


Trial 99 =========================================
15.929222010399386

[14.34670792 15.67862014 13.8877258  15.09973457 15.39483619 18.80679094
 14.37216319 16.24316207 12.45708805 13.86749982 16.23838797 15.92922201
 16.89900576 14.43946007 15.92922201 15.36797634 14.26222599 15.38566782
 18.8089845  15.67826647 16.04144271 17.51105334 16.16334951 15.38490755
 14.54691074 12.94727353 15.15672524 13.02489961 13.02111817 15.9910027
 13.41622859 17.20476715 16.12856209 15.37751087 14.32421884 16.18327771
 15.38528294 15.9813629  13.10972318 15.92922201 16.09990748 15.38017464
 15.92922201 17.49778175 15.92922201 16.20347685 14.27892101 14.51561002
 14.59381756 16.50887884 14.33771343 14.26407037 15.92922201 13.14589736
 15.40157117 13.80898918 16.23373765 15.39258615 15.37129379 16.09894935
 17.52445374 13.38606442 16.24278315 13.01893764 14.43877118 13.86728155
 14.9144673  16.24121162 16.07622967 17.58330786 16.24271558 15.92922201
 17.57676824 15.92922201 18.82050125 14.33611678 16.24

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.820501245998187
Avg = 15.437737602206619
Std = 1.3928384485001337


In [6]:
print(y_max_arr.tolist())

[14.346707921289035, 15.678620139789974, 13.887725801454867, 15.099734572016803, 15.394836186078374, 18.80679094329194, 14.372163187369043, 16.243162074973505, 12.457088048259141, 13.867499817983544, 16.23838797134717, 15.929222010399386, 16.899005758159696, 14.439460073052897, 15.929222010399386, 15.367976341817586, 14.262225993579378, 15.385667818208406, 18.80898450194613, 15.678266467037362, 16.041442705947006, 17.51105334419182, 16.163349514839236, 15.38490755472234, 14.546910737156905, 12.947273527434131, 15.156725238979995, 13.024899605185263, 13.021118168596654, 15.991002696358642, 13.416228592459412, 17.20476714613764, 16.128562087666737, 15.377510872060721, 14.3242188386987, 16.183277706094316, 15.38528293744223, 15.981362896346917, 13.109723176775884, 15.929222010399386, 16.099907482858576, 15.38017463601686, 15.929222010399386, 17.497781752770557, 15.929222010399386, 16.203476845954757, 14.278921006219827, 14.515610022003479, 14.593817557124368, 16.50887883950842, 14.3377134

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-1.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)